In [ ]:
import json
import importlib
import os
import sys
import json
from datetime import timedelta
# Rend le package src (à la racine du projet) importable depuis notebooks/
_project_root = os.path.abspath("..")
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from src.workflows.similar_tickets import (find_similar_tickets, group_similar_tickets, detect_similar_ticket_alert,run_workflow_sp, save_similar_ticket_alert)
from snowflake.snowpark.types import StringType
from snowflake.snowpark.context import get_active_session

In [ ]:
session = get_active_session()

## Test sur la similarité des tickets

In [ ]:
tickets = [
    {
        "ticket_id": "TEST-001",
        "received_at": "2026-10-06T09:00:00",
        "description": "The BAF09 surcharge displayed on SpotOn is incorrect.",
        "pol": "MXVER",
        "pod": "ESSCT"
    },
    {
        "ticket_id": "TEST-002",
        "received_at": "2026-10-06T10:00:00",
        "description": "SpotOn shows a wrong amount for the BAF09 surcharge.",
        "pol": "MXVER",
        "pod": "ESSCT"
    },
    {
        "ticket_id": "TEST-003",
        "received_at": "2026-10-06T11:00:00",
        "description": "SpotOn shows a wrong amount for the BAF09 surcharge.",
        "pol": "BDCGP",
        "pod": "MAAGA"
    },
    {
        "ticket_id": "TEST-004",
        "received_at": "2026-10-06T12:00:00",
        "description": "I cannot add a service to my pricelist.",
        "pol": "MXVER",
        "pod": "ESSCT"
    },
    {
        "ticket_id": "TEST-005",
        "received_at": "2026-10-05T09:00:00",
        "description": "The BAF09 surcharge displayed on SpotOn is incorrect.",
        "pol": "MXVER",
        "pod": "ESSCT"
    },
    {
        "ticket_id": "TEST-006",
        "received_at": "2026-10-06T13:00:00",
        "description": "The BAF09 charge amount shown on SpotOn is wrong.",
        "pol": "MXVER",
        "pod": "ESSCT"
    }
]


In [ ]:
similar = find_similar_tickets(
    tickets=tickets,
    reference_ticket=tickets[0],
    threshold=0.80
)

group = group_similar_tickets(
    reference_ticket=tickets[0],
    similar_tickets=similar,
    min_similarity=0.85,
    same_route=True
)


In [ ]:
print("=== TICKETS RETENUS ===")

for ticket in group["tickets"]:
    print(
        ticket["ticket_id"],
        "| Similarité :", ticket["similarity"]
    )

print("\n=== TICKETS EXCLUS ===")

retained_ids = {t["ticket_id"] for t in group["tickets"]}
similar_by_id = {t["ticket_id"]: t for t in similar}

ref = tickets[0]
ref_date = ref["received_at"][:10]

for ticket in tickets:
    tid = ticket["ticket_id"]

    if tid == ref["ticket_id"]:
        continue

    if tid in retained_ids:
        continue

    if ticket["received_at"][:10] != ref_date:
        reason = "Jour différent"

    elif tid not in similar_by_id:
        reason = "Similarité inférieure au seuil de détection (0.80)"
        # Le score n'a pas été conservé par find_similar_tickets()

    else:
        candidate = similar_by_id[tid]
        score = candidate["similarity"]

        if score < 0.85:
            reason = f"Similarité insuffisante ({score})"
        else:
            reason = "Route différente"

    print(tid, "|", reason)

## test du workflow indépendant

In [ ]:
alert = detect_similar_ticket_alert(
    tickets=tickets,
    reference_ticket=tickets[0],
    similarity_threshold=0.85,
    min_tickets=3
)

print("Alerte déclenchée :", alert["triggered"])
print("Nombre de tickets :", alert["total_tickets"])
print("Tickets concernés :", alert["ticket_ids"])
print("Message :", alert["message"])

In [ ]:
no_alert = detect_similar_ticket_alert(
    tickets=tickets,
    reference_ticket=tickets[0],
    similarity_threshold=0.85,
    min_tickets=5
)

print("Alerte déclenchée :", no_alert["triggered"])
print("Nombre de tickets :", no_alert["total_tickets"])
print("Message :", no_alert["message"])

In [ ]:
# Tables d'alertes

session.sql("""
CREATE TABLE IF NOT EXISTS PROJECT_DB.PUBLIC.SIMILAR_TICKET_ALERTS (
    ALERT_ID VARCHAR,
    CREATED_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    REFERENCE_TICKET_ID VARCHAR,
    TICKET_DATE DATE,
    TICKET_IDS VARIANT,
    TOTAL_TICKETS INTEGER,
    SIMILARITY_THRESHOLD FLOAT,
    MESSAGE TEXT,
    STATUS VARCHAR DEFAULT 'open'
)
""").collect()

In [ ]:
# Tables des exécutions

session.sql("""
CREATE TABLE IF NOT EXISTS PROJECT_DB.PUBLIC.WORKFLOW_RUNS (
    RUN_ID VARCHAR,
    RUN_AT TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    WORKFLOW_NAME VARCHAR,
    STATUS VARCHAR,
    TICKETS_ANALYZED INTEGER,
    ALERTS_CREATED INTEGER,
    DETAILS TEXT
)
""").collect()

In [ ]:
# Table de démonstration

session.sql("""
CREATE TABLE IF NOT EXISTS PROJECT_DB.PUBLIC.WORKFLOW_TICKETS (
    TICKET_ID VARCHAR,
    RECEIVED_AT VARCHAR,
    DESCRIPTION TEXT,
    POL VARCHAR,
    POD VARCHAR
)
""").collect()

In [ ]:
for ticket in tickets:
    session.sql("""
        MERGE INTO PROJECT_DB.PUBLIC.WORKFLOW_TICKETS t
        USING (
            SELECT
                ? AS TICKET_ID,
                ? AS RECEIVED_AT,
                ? AS DESCRIPTION,
                ? AS POL,
                ? AS POD
        ) s
        ON t.TICKET_ID = s.TICKET_ID
        WHEN NOT MATCHED THEN
            INSERT (TICKET_ID, RECEIVED_AT, DESCRIPTION, POL, POD)
            VALUES (
                s.TICKET_ID, s.RECEIVED_AT, s.DESCRIPTION,
                s.POL, s.POD
            )
    """, params=[
        ticket["ticket_id"],
        ticket["received_at"],
        ticket["description"],
        ticket.get("pol"),
        ticket.get("pod")
    ]).collect()

In [ ]:
session.sproc.register_from_file(
    file_path="../src/workflows/similar_tickets.py",
    func_name="run_workflow_sp",
    return_type=StringType(),
    input_types=[],
    name="PROJECT_DB.PUBLIC.RUN_SIMILAR_TICKETS",
    is_permanent=True,
    replace=True,
    stage_location="@PROJECT_DB.PUBLIC.FILES",
    packages=[
        "snowflake-snowpark-python",
        "snowflake-ml-python",
        "numpy"
    ]
)

In [ ]:
session.sql("""
    CALL PROJECT_DB.PUBLIC.RUN_SIMILAR_TICKETS()
""").show()

In [ ]:
session.sql("""
    SELECT *
    FROM PROJECT_DB.PUBLIC.SIMILAR_TICKET_ALERTS
    ORDER BY CREATED_AT DESC
""").show()

session.sql("""
    SELECT *
    FROM PROJECT_DB.PUBLIC.WORKFLOW_RUNS
    ORDER BY RUN_AT DESC
""").show()

# Workflow automatique

In [ ]:
session.sql("""
    CREATE OR REPLACE TASK PROJECT_DB.PUBLIC.TASK_SIMILAR_TICKETS
    USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE = 'XSMALL'
    SCHEDULE = '60 MINUTES'
    AS
    CALL PROJECT_DB.PUBLIC.RUN_SIMILAR_TICKETS()
""").collect()

In [ ]:
#Activation
session.sql("""
    ALTER TASK PROJECT_DB.PUBLIC.TASK_SIMILAR_TICKETS RESUME
""").collect()

#### On force l'execution pour ne pas attendre une heure

In [ ]:
session.sql("""
    EXECUTE TASK PROJECT_DB.PUBLIC.TASK_SIMILAR_TICKETS
""").collect()

In [ ]:
session.sql("""
    SELECT
        NAME,
        STATE,
        SCHEDULED_TIME,
        COMPLETED_TIME,
        ERROR_MESSAGE
    FROM TABLE(
        PROJECT_DB.INFORMATION_SCHEMA.TASK_HISTORY(
            TASK_NAME => 'TASK_SIMILAR_TICKETS',
            RESULT_LIMIT => 10
        )
    )
    ORDER BY SCHEDULED_TIME DESC
""").show()

# Test sur le scenarios.json

In [ ]:

with open("../tests/scenarios.json", "r", encoding="utf-8") as f:
    scenarios = json.load(f)["scenarios"]

for s in scenarios:
    print(s["id"], ":", s["ticket"]["description"][:80])

In [ ]:


sc = {s["id"]: s["ticket"] for s in scenarios}

# La date est récupérée depuis Snowflake :
# elle correspond à celle utilisée par la procédure automatique.
today = session.sql("""
    SELECT TO_CHAR(CURRENT_DATE(), 'YYYY-MM-DD') AS D
""").collect()[0]["D"]

s1 = sc["S1_question_documentaire"]
s2 = sc["S2_prix_plusieurs_donnees"]
s3 = sc["S3_cas_incertain"]

test_tickets = [
    {
        "ticket_id": "WF-S2-001",
        "received_at": f"{today}T09:00:00",
        "description": s2["description"],
        "pol": s2["pol"],
        "pod": s2["pod"]
    },
    {
        "ticket_id": "WF-S2-002",
        "received_at": f"{today}T09:30:00",
        "description": (
            "The customer sees an unexpected BAF09 surcharge "
            "of 150 per 40' on SpotOn for MXVER to ESSCT, "
            "but I expected this charge to be included in the freight. "
            "Why is the price different?"
        ),
        "pol": s2["pol"],
        "pod": s2["pod"]
    },
    {
        "ticket_id": "WF-S2-003",
        "received_at": f"{today}T10:00:00",
        "description": (
            "Why does SpotOn display a BAF09 surcharge of 150 "
            "for a 40' dry container from MXVER to ESSCT? "
            "The BAF09 should be included in the freight, "
            "so the displayed price is different from expected."
        ),
        "pol": s2["pol"],
        "pod": s2["pod"]
    },
    {
        "ticket_id": "WF-S1-001",
        "received_at": f"{today}T10:30:00",
        "description": s1["description"],
        "pol": s1["pol"],
        "pod": s1["pod"]
    },
    {
        "ticket_id": "WF-S3-001",
        "received_at": f"{today}T11:00:00",
        "description": s3["description"],
        "pol": s3["pol"],
        "pod": s3["pod"]
    }
]

for t in test_tickets:
    print(t["ticket_id"], "|", t["pol"], "→", t["pod"])

In [ ]:
for ticket in test_tickets:
    session.sql("""
        MERGE INTO PROJECT_DB.PUBLIC.WORKFLOW_TICKETS t
        USING (
            SELECT
                ? AS TICKET_ID,
                ? AS RECEIVED_AT,
                ? AS DESCRIPTION,
                ? AS POL,
                ? AS POD
        ) s
        ON t.TICKET_ID = s.TICKET_ID

        WHEN MATCHED THEN UPDATE SET
            t.RECEIVED_AT = s.RECEIVED_AT,
            t.DESCRIPTION = s.DESCRIPTION,
            t.POL = s.POL,
            t.POD = s.POD

        WHEN NOT MATCHED THEN
            INSERT (
                TICKET_ID, RECEIVED_AT, DESCRIPTION, POL, POD
            )
            VALUES (
                s.TICKET_ID, s.RECEIVED_AT, s.DESCRIPTION,
                s.POL, s.POD
            )
    """, params=[
        ticket["ticket_id"],
        ticket["received_at"],
        ticket["description"],
        ticket["pol"],
        ticket["pod"]
    ]).collect()

print("Tickets de test enregistrés.")

In [ ]:
alert = detect_similar_ticket_alert(
    tickets=test_tickets,
    reference_ticket=test_tickets[0],
    similarity_threshold=0.85,
    min_tickets=3
)

print("Alerte déclenchée :", alert["triggered"])
print("Tickets concernés :", alert["ticket_ids"])
print("Nombre :", alert["total_tickets"])

In [ ]:
session.sql("""
    CALL PROJECT_DB.PUBLIC.RUN_SIMILAR_TICKETS()
""").show()

In [ ]:
session.sql("""
    SELECT
        ALERT_ID,
        TICKET_IDS,
        TOTAL_TICKETS,
        MESSAGE,
        CREATED_AT
    FROM PROJECT_DB.PUBLIC.SIMILAR_TICKET_ALERTS
    WHERE TO_VARCHAR(TICKET_IDS) LIKE '%WF-S2-%'
    ORDER BY CREATED_AT DESC
""").show()